# Replicate dropout 0.2 across shared seeds

Reuse the context-thirty dropout-0.1 references for seeds 42, 43, and 44.
Reuse the completed seed-42 dropout-0.2 candidate. Train only dropout 0.2 for seeds 43 and 44.
Keep context thirty, width 16, budget 40/12, optimizer, data, sampling, loss, and participant folds fixed.
Both recipes have 2,884 parameters. Attention-weight dropout stays zero.
New work is ten fits and at most 400 epochs. OPAL remains disabled.
Assess paired macro-F1 and selected-checkpoint NLL beside REM gains, Wake behavior, and Deep recall.
A smaller train/validation gap caused only by higher training loss is not success.
Read docs/transformer-dropout-seeds.md before enabling preparation or training.

Completed results and investigation: docs/experiments/transformer-dropout-seeds-2026-11-11/README.md
Cell-by-cell guide: docs/notebook-19-guide.md


In [ ]:
import json
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("transformer_dropout_seed_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/transformer-dropout-seeds.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()
REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/transformer-dropout-seeds"  # Select the PR branch before merge
RUN_SETUP = False
RUN_CHECK = True
RUN_PREPARATION = False
RUN_TRAINING = False
RUN_CHECKPOINT_REPLAY = False
logger.info("Repository: %s", ROOT)
logger.info("Run setup: %s", RUN_SETUP)
logger.info("Run preparation: %s", RUN_PREPARATION)
logger.info("Run training: %s", RUN_TRAINING)

## Optional setup

Use the current checkout locally. Select experiment/transformer-dropout-seeds while reviewing this PR.
Use main after merge. Existing checkouts are not reset or switched automatically.
On a fresh machine, prepare the source cohort features through notebook 07 first.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")


## Preflight without training

Load development features and run a no-gradient forward pass.
The random model has no reported classification performance. No checkpoint or optimizer is created.


In [ ]:
if RUN_CHECK:
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -m sleep_classifier.neural_training \
        --config configs/transformer-dropout-02-seed-43.yaml --check
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Transformer preflight failed")

## Inspect the replication plan

Planning reads metadata and configurations only. It does not prepare files or train models.
Every new fold assignment must match seed 42. Only the training seed changes within each dropout recipe.


In [ ]:
import pandas as pd

from sleep_classifier.grouped_validation import GroupedValidationExperiment
from sleep_classifier.paired_architecture_report import PairedArchitectureReport
from sleep_classifier.paired_seed_report import PairedSeedReport

study = PairedSeedReport(ROOT / "configs/transformer-dropout-seeds.yaml")
plan = study.plan()
display(pd.DataFrame(plan["cases"]).set_index("seed"))
display(pd.DataFrame(plan["new_runs"])[["seed", "recipe", "folds", "maximum_epochs"]])
study.validate_reused_results()
logger.info("All reused artifacts passed hash, prediction, seed, and recipe checks")
logger.info("New fits: %s; maximum new epochs: %s", plan["new_fits"], plan["maximum_new_epochs"])
expected_folds = None
for case in study.cases:
    for role in ("reference", "candidate"):
        grouped_plan = GroupedValidationExperiment(Path(case[f"{role}_config"])).plan()
        if expected_folds is None:
            expected_folds = grouped_plan["folds"]
        if grouped_plan["folds"] != expected_folds:
            raise ValueError("Study folds differ from the measured seed-42 reference")
logger.info("All six grouped configurations use the same participant fold assignments")


## Prepare only the new dropout-0.2 folds

Enable RUN_PREPARATION once for fresh output directories.
All dropout-0.1 runs and the seed-42 dropout-0.2 run are reused. They are excluded from preparation and training.


In [ ]:
if RUN_PREPARATION:
    study.validate_reused_results()
    %cd "{ROOT}"
    for run in study.new_runs():
        grouped_config = run["config"]
        logger.info("Preparing dropout-0.2 seed %s", run["seed"])
        !uv run --locked --group gru --group notebook python \
            -m sleep_classifier.grouped_validation --config "{grouped_config}" --prepare
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Preparation failed; inspect the artifacts before retrying")
else:
    logger.info("Preparation skipped")


## Train the two new grouped candidates when ready

Enable RUN_TRAINING and execute the settings cell, then execute this cell interactively.
Runs are sequential: dropout-0.2 seed 43, followed by dropout-0.2 seed 44.
The first failed command stops the cell. Existing results are refused; automatic resume is not supported.


In [ ]:
if RUN_TRAINING:
    study.validate_reused_results()
    %cd "{ROOT}"
    for run in study.new_runs():
        grouped_config = run["config"]
        logger.info("Training dropout-0.2 seed %s", run["seed"])
        !uv run --locked --group gru --group notebook python -u \
            -m sleep_classifier.grouped_validation --config "{grouped_config}" --train
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Training failed; completed artifacts are retained")
else:
    logger.info("Training skipped: enable RUN_TRAINING when ready")


## Inspect complete and pending pairs

Each completed pair is independently verified from its saved predictions.
An existing seed-42 result does not mean the new seeds have run.
The full study report requires all three pairs to finish.


In [ ]:
statuses = study.status()
display(pd.DataFrame(statuses).set_index(["seed", "recipe"]))
seed_rows = []
class_rows = []
fold_rows = []
for case in study.cases:
    pair_statuses = [row for row in statuses if row["seed"] == case["seed"]]
    if not all(row["status"] == "complete" for row in pair_statuses):
        continue
    paired = PairedArchitectureReport(
        Path(case["reference_summary"]), Path(case["candidate_summary"]),
        comparison=study.comparison,
    ).build()
    reference_mean = pd.DataFrame(paired["folds"])["reference_macro_f1"].mean()
    candidate_mean = pd.DataFrame(paired["folds"])["candidate_macro_f1"].mean()
    seed_rows.append({"seed": case["seed"], "reference_mean_macro_f1": reference_mean,
                      "candidate_mean_macro_f1": candidate_mean,
                      "paired_gain": paired["paired"]["mean_macro_f1_difference"],
                      "participants_improved": sum(row["macro_f1_difference"] > 0
                                                   for row in paired["participants"])})
    for role in ("reference", "candidate"):
        for stage, scores in paired[role]["classes"].items():
            class_rows.append({"seed": case["seed"], "recipe": role,
                               "stage": ["Wake", "Light", "Deep", "REM"][int(stage)], **scores})
    fold_rows.extend({"seed": case["seed"], **row} for row in paired["folds"])
if seed_rows:
    display(pd.DataFrame(seed_rows).set_index("seed"))
    display(pd.DataFrame(class_rows).set_index(["seed", "recipe", "stage"]))
    display(pd.DataFrame(fold_rows).set_index(["seed", "fold"]))


In [ ]:
if all(row["status"] == "complete" for row in statuses):
    result = study.build()
    study.output_dir.mkdir(parents=True, exist_ok=True)
    output = study.output_dir / "summary.json"
    output.write_text(json.dumps(result, indent=2) + "\n")
    display(pd.DataFrame([{
        "mean_seed_paired_gain": result["mean_seed_macro_f1_difference"],
        "sample_std_seed_paired_gain": result["sample_std_seed_macro_f1_difference"],
    }]))
    logger.info("Validated seed comparison saved: %s", output)
else:
    logger.info("Study incomplete: seed-level aggregate requires all three completed pairs")


In [ ]:
if seed_rows:
    import matplotlib.pyplot as plt

    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    seed_table = pd.DataFrame(seed_rows).set_index("seed")
    seed_table[["reference_mean_macro_f1", "candidate_mean_macro_f1"]].plot.bar(
        ax=axes[0], ylim=(0, 1),
    )
    axes[0].set(xlabel="Training seed", ylabel="Mean development fold macro-F1")
    classes = pd.DataFrame(class_rows)
    rem = classes[classes["stage"] == "REM"].pivot(index="seed", columns="recipe", values="recall")
    rem.plot.bar(ax=axes[1], ylim=(0, 1))
    axes[1].set(xlabel="Training seed", ylabel="Pooled development REM recall")
    figure.tight_layout()
    plt.show()


In [ ]:
if class_rows:
    import matplotlib.pyplot as plt

    classes = pd.DataFrame(class_rows)
    figure, axes = plt.subplots(1, 3, figsize=(14, 4))
    for axis, stage in zip(axes, ["Wake", "Deep", "REM"]):
        stage_scores = classes[classes["stage"] == stage].pivot(
            index="seed", columns="recipe", values="f1",
        )
        stage_scores.plot.bar(ax=axis, ylim=(0, 1))
        axis.set(title=stage, xlabel="Training seed", ylabel="Pooled development F1")
    figure.tight_layout()
    plt.show()


## Inspect selected-checkpoint probability losses

Cross-entropy penalizes confident errors even when macro-F1 improves.
This post-hoc diagnostic recomputes NLL from saved probabilities at each selected checkpoint.
It describes development predictions; confidence on errors alone does not establish calibration.


In [ ]:
import numpy as np

probability_diagnostics = []
for case in study.cases:
    if not all(row["status"] == "complete" for row in statuses if row["seed"] == case["seed"]):
        continue
    for role in ("reference", "candidate"):
        saved = json.loads(Path(case[f"{role}_summary"]).read_text())
        prediction_frames = []
        for result in saved["results"]:
            path = Path(result["metrics_path"])
            if not path.is_absolute():
                path = Path(case[f"{role}_summary"]).parent / path
            prediction_frames.append(pd.read_csv(path.parent / "validation_predictions.csv"))
        predictions = pd.concat(prediction_frames, ignore_index=True)
        probabilities = predictions[[f"probability_{stage}" for stage in range(4)]].to_numpy()
        if (not np.isfinite(probabilities).all() or (probabilities < 0).any()
                or (probabilities > 1).any() or not np.allclose(probabilities.sum(axis=1), 1)):
            raise ValueError("Saved class probabilities are invalid")
        targets = predictions.target.to_numpy(dtype=int)
        wrong = probabilities.argmax(axis=1) != targets
        confidence = probabilities.max(axis=1)
        probability_diagnostics.append({
            "seed": case["seed"], "recipe": role, "epochs": len(predictions),
            "checkpoint_nll": float(-np.log(np.maximum(
                probabilities[np.arange(len(targets)), targets], 1e-12,
            )).mean()),
            "mean_confidence_correct": float(confidence[~wrong].mean()) if (~wrong).any() else None,
            "mean_confidence_wrong": float(confidence[wrong].mean()) if wrong.any() else None,
        })
if probability_diagnostics:
    display(pd.DataFrame(probability_diagnostics).set_index(["seed", "recipe"]))


## Inspect the reported loss rise near epoch fourteen

Compare all seeds and both dropout recipes for folds 01 and 04.
The epoch-fourteen line is a visual reference, not a configured training event.
The table separates minimum NLL, best macro-F1, and the largest upward loss step.
Only scalar histories were saved per epoch. These cannot identify which participant or class caused an old loss jump.
See docs/notebook-19-guide.md for every cell and the investigation limits.


In [ ]:
if all(row["status"] == "complete" for row in statuses):
    import matplotlib.pyplot as plt

    epoch_audit = []
    figure, axes = plt.subplots(3, 2, figsize=(12, 10))
    for row_index, case in enumerate(study.cases):
        for role in ("reference", "candidate"):
            path = Path(case[f"{role}_summary"])
            if not path.exists():
                continue
            saved = json.loads(path.read_text())
            for column_index, fold in enumerate(["fold-01", "fold-04"]):
                result = next((entry for entry in saved["results"] if entry["fold"] == fold), None)
                if result is None:
                    continue
                metrics_path = Path(result["metrics_path"])
                if not metrics_path.is_absolute():
                    metrics_path = path.parent / metrics_path
                metrics = json.loads(metrics_path.read_text())
                history = pd.DataFrame(metrics["history"])
                minimum = history.loc[history["validation_loss"].idxmin()]
                jumps = history["validation_loss"].diff()
                largest = history.loc[jumps.idxmax()]
                epoch_audit.append({
                    "seed": case["seed"], "recipe": role, "fold": fold,
                    "minimum_nll_epoch": int(minimum["epoch"]),
                    "minimum_nll": float(minimum["validation_loss"]),
                    "best_macro_f1_epoch": metrics["best_epoch"],
                    "largest_upward_step_epoch": int(largest["epoch"]),
                    "largest_upward_step": float(jumps.max()),
                    "epochs_10_to_18": history[history["epoch"].between(10, 18)].to_dict("records"),
                })
                axis = axes[row_index, column_index]
                axis.plot(history["epoch"], history["validation_loss"], label=role)
                axis.set(title=f"Seed {case['seed']}, {fold}",
                         xlabel="Epoch", ylabel="Validation NLL")
    for axis in axes.flat:
        axis.axvline(14, color="gray", linestyle=":", label="Epoch 14 reference line")
        axis.legend(fontsize=8)
    figure.tight_layout()
    plt.show()
    display(pd.DataFrame(epoch_audit).drop(columns="epochs_10_to_18"))
    audit_path = study.output_dir / "epoch-audit.json"
    audit_path.parent.mkdir(parents=True, exist_ok=True)
    audit_path.write_text(json.dumps(epoch_audit, indent=2) + "\n")
    replay_path = study.output_dir / "checkpoint-replay.json"
    if replay_path.exists():
        display(pd.DataFrame(json.loads(replay_path.read_text())))
    else:
        logger.info("Optional checkpoint replay has not run")
else:
    logger.info("The focused epoch audit requires every grouped run to be complete")


## Optional read-only selected-checkpoint replay

This tests seed-42 selected checkpoints for folds 01 and 04, not missing intermediate checkpoints.
It restores stored training-fitted scaling and evaluates each checkpoint twice without gradients or optimizer updates.
RUN_CHECKPOINT_REPLAY defaults to False. The local probe used MPS, matching the saved runs.
HTML export skips this cell and displays existing replay results in the preceding audit cell.


In [ ]:
if RUN_CHECKPOINT_REPLAY:
    from sleep_classifier.checkpoint_audit import CheckpointReplayAudit

    replay_rows = []
    for fold in ["fold-01", "fold-04"]:
        directory = ROOT / "artifacts/transformer-dropout-grouped-validation" / fold
        replay_rows.append(CheckpointReplayAudit(directory, device="mps").build())
    replay_path = study.output_dir / "checkpoint-replay.json"
    replay_path.parent.mkdir(parents=True, exist_ok=True)
    replay_path.write_text(json.dumps(replay_rows, indent=2) + "\n")
    display(pd.DataFrame(replay_rows))
else:
    logger.info("Checkpoint replay skipped; it never starts optimization")


## Export HTML without training

The export runs the no-gradient preflight and report cells.
It skips setup, preparation, training, and this export command.


In [ ]:
!uv run --locked --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/19_transformer_dropout_seeds.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Seed report export failed")
